# SteelVision — 01 · YOLOv8n baseline on NEU-DET

Run on **Kaggle** with GPU (Settings → Accelerator → GPU T4 x2, use one GPU).

**Before running**
1. The notebook clones the public repo `Minhmaxxx/SteelVision` — make sure the repo is public (or use a Kaggle secret for a token).
2. Optional but recommended: Add-ons → Secrets → add `WANDB_API_KEY` to log experiments to Weights & Biases. Without it the run still works — W&B logging is skipped and metrics land in `runs/.../results.csv`.

Expected runtime: ~30–60 min for 100 epochs of YOLOv8n @ 640px on NEU-DET.

In [ ]:
%pip install -q ultralytics wandb kagglehub

REPO_URL = "https://github.com/Minhmaxxx/SteelVision.git"

In [ ]:
import os
import sys
import zipfile
from pathlib import Path

if Path("steelvision_src.zip").exists():  # uploaded alongside the kernel
    zipfile.ZipFile("steelvision_src.zip").extractall("steelvision_src")
    sys.path.insert(0, str(Path("steelvision_src").resolve()))
else:
    # Fallback: clone the repo (works only if the repo is public).
    os.environ["GIT_TERMINAL_PROMPT"] = "0"  # fail fast instead of hanging
    !git clone --depth 1 {REPO_URL} repo_clone
    sys.path.insert(0, str(Path("repo_clone/src").resolve()))

import steelvision
print("steelvision", steelvision.__version__)

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "No CUDA device in this Kaggle session — the GPU config did not apply "
        f"(device_count={torch.cuda.device_count()}). Fix kernel accelerator metadata."
    )
print("GPU in session:", torch.cuda.get_device_name(0))

In [ ]:
# Download NEU-DET from Kaggle, convert VOC XML -> YOLO, dedupe, split.
DATA_ROOT = "/kaggle/working/data"
!python -m steelvision.data.prepare --datasets neu_det --data-root {DATA_ROOT}
DATA_YAML = f"{DATA_ROOT}/processed/neu_det/data.yaml"
print(DATA_YAML)

In [ ]:
import os

import wandb

try:  # optional: add WANDB_API_KEY via Kaggle Add-ons -> Secrets
    from kaggle_secrets import UserSecretsClient
    os.environ.setdefault("WANDB_API_KEY", UserSecretsClient().get_secret("WANDB_API_KEY"))
except Exception:
    pass

if os.environ.get("WANDB_API_KEY"):
    wandb.login()
else:
    # Headless run without W&B: metrics still land in runs/.../results.csv
    os.environ["WANDB_MODE"] = "disabled"
    os.environ["WANDB_DISABLED"] = "true"

from ultralytics import YOLO

model = YOLO("yolov8n.pt")
model.train(
    data=DATA_YAML,
    epochs=100,
    imgsz=640,
    batch=16,
    seed=42,
    device=0,  # single GPU; use "0,1" to train on both T4s
    project="/kaggle/working/runs",
    name="yolov8n-neu-det-baseline",
    exist_ok=True,
)

In [ ]:
import json

metrics = model.val(
    split="test",
    project="/kaggle/working/runs",
    name="yolov8n-neu-det-baseline-test",
    exist_ok=True,
)
summary = {k: float(v) for k, v in metrics.results_dict.items()}
print(summary)
try:
    summary["per_class_ap50"] = {
        metrics.names[i]: float(ap) for i, ap in enumerate(metrics.box.ap50)
    }
    print(summary["per_class_ap50"])
except AttributeError:
    pass
json.dump(summary, open("/kaggle/working/test_metrics.json", "w"), indent=2)

In [ ]:
!cd /kaggle/working && zip -qr yolov8n-neu-det-baseline.zip runs test_metrics.json
from IPython.display import FileLink
FileLink("yolov8n-neu-det-baseline.zip")

## Next ablations (one variable at a time)

| # | Change | Where |
|---|---|---|
| 1 | `imgsz=1024` (small defects) | `model.train(...)` |
| 2 | Disable mosaic/mixup augmentation (`mosaic=0.0, mixup=0.0`) | `model.train(...)` |
| 3 | Model size `yolov8s.pt` | `YOLO(...)` |
| 4 | GC10-DET training run (`--datasets gc10`) | data prep cell |

Copy this notebook per ablation (Kaggle → Save & Run All → Version), record every
run in W&B, and put final numbers into `docs/reports/experiments.vi.md`.